# Detect and identify fossil specimens

This demonstration uses saved RF-DETR, Swin and DINO models on verified validation examples. It performs no training or test-set evaluation. The region starts as a prepared RGB image; whole-slide focal selection and tiling are separate preprocessing stages.

**Before running:** follow [installation and artifact retrieval](README.md). Copy `config/demo.example.json` to `config/local.demo.json`, set your paths, and select the `Smithsonian review` kernel. Cells execute from top to bottom. Each run uses a new output directory. Missing or mismatched artifacts cause an explicit error.

In [ ]:
import os
from pathlib import Path
from uuid import uuid4
from IPython.display import display, HTML
from PIL import Image
from smithsonian.configuration.paths import repository_root
from smithsonian.configuration.options import save_effective_config
from smithsonian.inference.demo import (
    prepare_demo, classify_example, detect_example,
    annotate_predictions, prediction_table,
)

config_path = Path(os.environ.get("SMITHSONIAN_DEMO_CONFIG", repository_root() / "config/local.demo.json"))
config, sample = prepare_demo(config_path)
output = Path(config["output"]) / uuid4().hex[:12]
output.mkdir(parents=True, exist_ok=False)
save_effective_config(config, output / "effective_config.json")
print("Verified artifacts. Outputs:", output)

## 1. A real specimen and its expert reference
The reference is the expert annotation; model predictions may differ.

In [ ]:
print("Reference:", sample["category"])
print("Source record:", sample["record_id"])
with Image.open(config["specimen"]) as image:
    display(image.copy())

## 2. Saved Swin and DINO classifiers
Both receive the same crop. Scores are model probabilities, not a measure of overall accuracy.

In [ ]:
specimen_results = {
    model: classify_example(config, model, output / f"specimen_{model}.json")
    for model in ("swin", "dino")
}
for model, result in specimen_results.items():
    row = result["predictions"][0]
    print(f"{model}: {row['category']} ({row['score']:.1%})")

## 3. Detect specimens in a region
RF-DETR returns boxes in this image’s pixel coordinates. The same boxes will be used for both classifiers.

In [ ]:
detections = detect_example(config, output / "detection")
print("Detected specimens:", len(detections["predictions"]))
with Image.open(config["image"]) as image:
    display(image.copy())

## 4. Crop and identify each detection
The package adds the fixed 10% context margin, pads in white and resizes to 224×224 pixels. Swin and DINO are alternative classifiers.

In [ ]:
from smithsonian.preprocessing.crops import crop_input

with Image.open(config["image"]) as source:
    region = source.convert("RGB")
    for detection in detections["predictions"][:6]:
        display(crop_input(region, detection["box"]))


In [ ]:
combined_results = {
    model: classify_example(
        config, model, output / f"combined_{model}.json",
        image=config["image"], boxes=output / "detection/boxes.json",
    )
    for model in ("swin", "dino")
}
for model, result in combined_results.items():
    print(model.upper())
    display(annotate_predictions(config["image"], result))

## 5. Output table
These are predictions on one region, not an evaluation statistic. JSON outputs retain the full probabilities and input hashes.

In [ ]:
from html import escape
import csv

rows = prediction_table(combined_results)
columns = ["model", "specimen", "category", "score", "box"]
with (output / "predictions.csv").open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=columns)
    writer.writeheader()
    writer.writerows(rows)
header = "".join(f"<th>{escape(key)}</th>" for key in columns)
body = "".join("<tr>" + "".join(f"<td>{escape(str(row[key]))}</td>" for key in columns) + "</tr>" for row in rows)
display(HTML(f"<table><tr>{header}</tr>{body}</table>"))
print("Saved", output / "predictions.csv")